# Juman++ で形態素解析する（代表表記・カテゴリ付き）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/jumanpp.ipynb)

NLP クックブック 03-24 ／ 日本語の形態素解析と文分割

単語の区切りと品詞に加えて、「りんご/林檎/リンゴ」をまとめる代表表記や、「人」「場所」のような意味カテゴリも欲しい。京都大学の研究室が開発した、RNN 言語モデルで単語列のもっともらしさを評価する形態素解析器 Juman++ を使う。

In [1]:
# ダウンロード（約 300MB）・展開・ビルドで、Colab（2 コア）では 10 分ほどかかる見込み
!wget -q https://github.com/ku-nlp/jumanpp/releases/download/v2.0.0-rc4/jumanpp-2.0.0-rc4.tar.xz
!tar xf jumanpp-2.0.0-rc4.tar.xz
!cd jumanpp-2.0.0-rc4 && mkdir -p bld && cd bld && cmake .. -DCMAKE_BUILD_TYPE=Release -DJPP_ENABLE_TESTS=OFF -DCMAKE_INSTALL_PREFIX=/usr/local > /dev/null 2>&1 && make install -j2 > /dev/null 2>&1
!jumanpp --version

Juman++ Version: 2.0.0-rc4 / Dictionary: 20210915-2a1662a / LM: K:20210809-0011280 L:20210116-10aab9fb F:20171214-9d125cb


In [2]:
!pip install -q rhoknp==1.8.2 fugashi unidic-lite

In [3]:
import subprocess

# 生の出力（JUMAN 形式）: 表層形 読み 原形 品詞 品詞ID 細分類 ID 活用型 ID 活用形 ID 意味情報
r = subprocess.run(["jumanpp"], input="外国人参政権について話し合った。\n", capture_output=True, text=True)
print(r.stdout)

外国 がいこく 外国 名詞 6 普通名詞 1 * 0 * 0 "代表表記:外国/がいこく ドメイン:政治 カテゴリ:場所-その他"
人 じん 人 名詞 6 普通名詞 1 * 0 * 0 "代表表記:人/じん カテゴリ:人 漢字読み:音"
参政 さんせい 参政 名詞 6 サ変名詞 2 * 0 * 0 "代表表記:参政/さんせい ドメイン:政治 カテゴリ:抽象物"
権 けん 権 名詞 6 普通名詞 1 * 0 * 0 "代表表記:権/けん カテゴリ:抽象物 漢字読み:音"
に に に 助詞 9 格助詞 1 * 0 * 0 "代表表記:に/に"
ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:付く/つく 自他動詞:他:付ける/つける 補文ト 付属動詞候補（基本）"
@ ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:就く/つく ドメイン:ビジネス 自他動詞:他:就ける/つける"
@ ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:搗く/つく ドメイン:料理・食事"
@ ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:点く/つく 自他動詞:他:点ける/つける"
@ ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:着く/つく ドメイン:交通 自他動詞:他:着ける/つける"
@ ついて ついて つく 動詞 2 * 0 子音動詞カ行 2 タ系連用テ形 14 "代表表記:突く/つく"
話し はなし 話す 動詞 2 * 0 子音動詞サ行 5 基本連用形 8 "代表表記:話す/はなす 補文ト"
合った あった 合う 動詞 2 * 0 子音動詞ワ行 12 タ形 10 "代表表記:合う/あう 反義:動詞:離れる/はなれる 自他動詞:他:合わす/あわす 付属動詞候補（基本）"
。 。 。 特殊 1 句点 1 * 0 * 0 "代表表記:。/。"
EOS



In [4]:
from rhoknp import Jumanpp

# skip_sanity_check=True: 起動時の確認（空の文を送る）を省く（下の注意書きを参照）
# 別の場所に入れたときは Jumanpp("/path/to/bin/jumanpp", skip_sanity_check=True)。options=["--beam", "10"] でオプションも渡せる
jumanpp = Jumanpp(skip_sanity_check=True)

def analyze(text, keys=("代表表記", "カテゴリ", "ドメイン")):
    for m in jumanpp.apply_to_sentence(text).morphemes:
        sem = {k: v for k, v in m.semantics.items() if keys is None or k in keys}
        print(m.text, m.reading, m.lemma, m.pos, m.subpos, m.conjtype, m.conjform, sem, sep="\t")

analyze("国境の長いトンネルを抜けると雪国であった。")
print()
analyze("スマホでググった。", keys=None)   # 意味情報をすべて表示
print()

# 生の出力の @ 行（同形異義語）は homographs に入る
for m in jumanpp.apply_to_sentence("外国人参政権について話し合った。").morphemes:
    if m.homographs:
        print(m.text, m.canon, [h.canon for h in m.homographs])

国境	こっきょう	国境	名詞	普通名詞	*	*	{'代表表記': '国境/こっきょう', 'カテゴリ': '場所-その他'}
の	の	の	助詞	格助詞	*	*	{'代表表記': 'の/の'}
長い	ながい	長い	形容詞	*	イ形容詞アウオ段	基本形	{'代表表記': '長い/ながい'}
トンネル	とんねる	トンネル	名詞	サ変名詞	*	*	{'代表表記': 'トンネル/とんねる', 'ドメイン': '交通', 'カテゴリ': '場所-施設'}
を	を	を	助詞	格助詞	*	*	{'代表表記': 'を/を'}
抜ける	ぬける	抜ける	動詞	*	母音動詞	基本形	{'代表表記': '抜ける/ぬける'}
と	と	と	助詞	格助詞	*	*	{'代表表記': 'と/と'}
雪国	ゆきぐに	雪国	名詞	普通名詞	*	*	{'代表表記': '雪国/ゆきぐに', 'カテゴリ': '場所-その他'}
であった	であった	だ	判定詞	*	判定詞	デアル列タ形	{'代表表記': 'だ/だ'}
。	。	。	特殊	句点	*	*	{'代表表記': '。/。'}

スマホ	スマホ	スマホ	名詞	普通名詞	*	*	{'自動獲得': 'Wikipedia', 'Wikipediaリダイレクト': 'スマートフォン'}
で	で	で	助詞	格助詞	*	*	{'代表表記': 'で/で'}
ググった	ぐぐった	ググる	動詞	*	子音動詞ラ行	タ形	{'代表表記': 'ググる/ぐぐる', '自動獲得': 'テキスト'}
。	。	。	特殊	句点	*	*	{'代表表記': '。/。'}

ついて 付く/つく ['就く/つく', '搗く/つく', '点く/つく', '着く/つく', '突く/つく']


In [5]:
# 代表表記 (canon) で表記ゆれをまとめる。代表表記がない語（自動獲得語など）は canon が None なので表層形で代用する
from collections import Counter

texts = ["りんごを食べた。", "林檎を食べた。", "リンゴを食べた。", "子どもと子供とこどもが遊ぶ。"]
surf, canon = Counter(), Counter()
for t in texts:
    for m in jumanpp.apply_to_sentence(t).morphemes:
        if m.pos == "名詞":
            surf[m.text] += 1
            canon[m.canon or m.text] += 1
print("表層形:  ", surf.most_common())
print("代表表記:", canon.most_common())

表層形:   [('りんご', 1), ('林檎', 1), ('リンゴ', 1), ('子ども', 1), ('子供', 1), ('こども', 1)]
代表表記: [('林檎/りんご', 3), ('子供/こども', 3)]


In [6]:
# MeCab + UniDic (fugashi) と、分割と品詞を比べる（例文はこの章の「解析器・辞書による分割の違いを比べる」と同じ）
from fugashi import Tagger

unidic = Tagger()
for s in ["外国人参政権について話し合った。", "東京都立大学で機械翻訳を研究しています。", "食べさせられたくなかった。"]:
    print(s)
    print("  Juman++     ", " | ".join(f"{m.text}/{m.pos}" for m in jumanpp.apply_to_sentence(s).morphemes))
    print("  MeCab+UniDic", " | ".join(f"{w.surface}/{w.feature.pos1}" for w in unidic(s)))

外国人参政権について話し合った。
  Juman++      外国/名詞 | 人/名詞 | 参政/名詞 | 権/名詞 | に/助詞 | ついて/動詞 | 話し/動詞 | 合った/動詞 | 。/特殊
  MeCab+UniDic 外国/名詞 | 人参/名詞 | 政権/名詞 | に/助詞 | つい/動詞 | て/助詞 | 話し合っ/動詞 | た/助動詞 | 。/補助記号
東京都立大学で機械翻訳を研究しています。
  Juman++      東京/名詞 | 都立/名詞 | 大学/名詞 | で/助詞 | 機械/名詞 | 翻訳/名詞 | を/助詞 | 研究/名詞 | して/動詞 | い/接尾辞 | ます/接尾辞 | 。/特殊
  MeCab+UniDic 東京/名詞 | 都立/名詞 | 大学/名詞 | で/助詞 | 機械/名詞 | 翻訳/名詞 | を/助詞 | 研究/名詞 | し/動詞 | て/助詞 | い/動詞 | ます/助動詞 | 。/補助記号
食べさせられたくなかった。
  Juman++      食べ/動詞 | させ/接尾辞 | られ/接尾辞 | たく/接尾辞 | なかった/接尾辞 | 。/特殊
  MeCab+UniDic 食べ/動詞 | させ/助動詞 | られ/助動詞 | たく/助動詞 | なかっ/形容詞 | た/助動詞 | 。/補助記号


- 品詞体系と辞書は JUMAN のもので、UniDic とは品詞名も単位も違う。「であった」は判定詞 1 語、「させ」「られ」「たく」「なかった」は接尾辞になる（UniDic では助動詞や形容詞で、「なかっ/た」と切れる）。構文解析の章の「KWJA で文節・基本句の係り受けと並列構造を解析する」の KWJA も同じ品詞体系で、結果は rhoknp で同じように扱える。
- 「外国人参政権」は、Juman++ では「外国/人/参政/権」、MeCab + UniDic では「外国/人参/政権」になった。Juman++ は単語列の意味的なもっともらしさを RNN 言語モデルで考慮する（README から張られたウェブデモもこの文を例にしている）。比べた 3 文は、この章の「解析器・辞書による分割の違いを比べる」と同じ例文なので、Sudachi や Janome の結果とも見比べられる。
- 意味情報には、代表表記、カテゴリ（人、場所-施設、抽象物など）、ドメイン（政治、料理・食事など）、自他動詞や反義語の対応が入る。代表表記は「Sudachi の正規化形で表記ゆれを吸収する」の正規化形と同じ目的に使え、「りんご/林檎/リンゴ」は「林檎/りんご」に、「子ども/子供/こども」は「子供/こども」にまとまった。
- 生の出力の `@` で始まる行は、読みも品詞も同じで代表表記だけが違う候補（同形異義語）。「スマホ」のような自動獲得語は代表表記がなく（`canon` が None）、意味情報に「Wikipediaリダイレクト:スマートフォン」が入った。
- Juman++ (v2.0.0-rc4) は、最初の入力行が空行だと異常終了（セグメンテーション違反）することがある（毎回ではない）。rhoknp (1.8.2) の `Jumanpp()` は起動時の確認で空の文を送るので、異常終了した回は「failed to start Juman++」という警告が出る（解析は再起動したプロセスで行われる）。`skip_sanity_check=True` で確認を省いたときは、最初に空文字列を渡さない。subprocess で直接渡すときも先頭に空行を置かない。
- 配布物（約 300MB）に学習済みのモデルが含まれるので、GitHub のソースではなくリリースの tar.xz を使う。ライセンスは Apache 2.0。最新版は 2023 年の v2.0.0-rc4。